# CipherStop - LSTM + Attention model (KAN-24 / KAN-25 / KAN-26)

Early ransomware detection from variable-length behavioral sequences.

- **Input**: `(batch, seq_len, n_features)` + `lengths` `(batch,)`
- **Output**: detection score in `[0, 1]` per sequence (1 = ransomware)

## 1. Imports

In [ ]:
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

## 2. Attention layer (KAN-25)
Additive attention over LSTM outputs. Padded steps are masked with `-inf`, so they get exactly 0 weight.

In [ ]:
class Attention(nn.Module):
    """Additive attention over the LSTM outputs.

    For each time step t:  e_t = v^T tanh(W h_t)
    Weights: alpha = softmax(e) over the *real* (non-padded) steps only.
    Context vector: sum_t alpha_t * h_t
    """

    def __init__(self, hidden_dim: int, attn_dim: int = 64):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(
        self, h: torch.Tensor, mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # h: (B, T, H), mask: (B, T) True for real steps
        scores = self.v(torch.tanh(self.proj(h))).squeeze(-1)  # (B, T)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)  # (B, T), padding -> 0
        context = torch.bmm(weights.unsqueeze(1), h).squeeze(1)  # (B, H)
        return context, weights

## 3. LSTM + Attention model (KAN-24)
Unidirectional on purpose: early detection must only use the past.
Uses `pack_padded_sequence` so variable-length sequences are handled correctly.

In [ ]:
class RansomwareLSTM(nn.Module):
    """LSTM + attention binary classifier for variable-length sequences.

    The LSTM is unidirectional on purpose: for early detection the model
    must only use the past, exactly like in real-time monitoring.
    """

    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
        attn_dim: int = 64,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = Attention(hidden_dim, attn_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)

    def logits(
        self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Raw scores before sigmoid (use these with BCEWithLogitsLoss)."""
        B, T, _ = x.shape
        if lengths is None:
            lengths = torch.full((B,), T, dtype=torch.long)
        lengths = lengths.to("cpu").clamp(min=1, max=T)

        packed = pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)

        mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.to(
            x.device
        ).unsqueeze(1)
        context, weights = self.attention(out, mask)
        logit = self.classifier(self.dropout(context)).squeeze(-1)  # (B,)
        return logit, weights

    def forward(
        self,
        x: torch.Tensor,
        lengths: Optional[torch.Tensor] = None,
        return_attention: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """Returns detection score in [0, 1], shape (B,)."""
        logit, weights = self.logits(x, lengths)
        score = torch.sigmoid(logit)
        return (score, weights) if return_attention else score

## 4. Quick demo

In [ ]:
N_FEATURES = 24  # replace with len(feature_cols) from the Spark pipeline

torch.manual_seed(0)
model = RansomwareLSTM(n_features=N_FEATURES, hidden_dim=32, num_layers=2).eval()

x = torch.randn(3, 40, N_FEATURES)
lengths = torch.tensor([40, 25, 5])
score, weights = model(x, lengths, return_attention=True)
print('scores :', score)
print('scores shape:', score.shape, '| attention shape:', weights.shape)

## 5. Unit tests for the forward pass (KAN-26)
Each test is a plain function with `assert`s; the last cell runs them all.

In [ ]:
def make_model():
    torch.manual_seed(0)
    return RansomwareLSTM(n_features=N_FEATURES, hidden_dim=32, num_layers=2).eval()

In [ ]:
def test_output_shape_and_range():
    model = make_model()
    score = model(torch.randn(4, 50, N_FEATURES))
    assert score.shape == (4,)
    assert torch.all(score >= 0) and torch.all(score <= 1)


def test_variable_length_batch():
    model = make_model()
    score = model(torch.randn(3, 40, N_FEATURES), torch.tensor([40, 25, 5]))
    assert score.shape == (3,)
    assert torch.isfinite(score).all()


def test_padding_does_not_change_score():
    model = make_model()
    seq = torch.randn(1, 20, N_FEATURES)
    padded = torch.cat([seq, torch.randn(1, 30, N_FEATURES) * 100], dim=1)
    s1 = model(seq, torch.tensor([20]))
    s2 = model(padded, torch.tensor([20]))
    assert torch.allclose(s1, s2, atol=1e-5)


def test_different_sequence_lengths_accepted():
    model = make_model()
    for T in (1, 10, 200):
        assert model(torch.randn(2, T, N_FEATURES)).shape == (2,)


def test_attention_weights_sum_to_one():
    model = make_model()
    _, w = model(torch.randn(2, 30, N_FEATURES), torch.tensor([30, 12]), return_attention=True)
    assert torch.allclose(w.sum(dim=1), torch.ones(2), atol=1e-5)
    assert w[1, 12:].abs().sum() == 0  # no attention on padding


def test_gradients_flow():
    model = make_model()
    model.train()
    model(torch.randn(4, 30, N_FEATURES)).sum().backward()
    assert all(p.grad is not None for p in model.parameters())

In [ ]:
tests = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
for t in tests:
    t()
    print(f"PASSED  {t.__name__}")
print(f"\n{len(tests)} passed")